# Tətbiqi məlumat analitikası: Etibarlılıq intervalları və Hipotezlərin yoxlanılması

Bu praktiki məşğələnin məqsədi, populyasiya parametrlərini qiymətləndirmək və iddiaları statistik cəhətdən əsaslandırmaq üçün Python istifadəsini mənimsəməkdir. Tapşırıqlara başlamazdan əvvəl aşağıdakı kitabxanaları və datasetləri yükləyin.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Datasetlərin yüklənməsi
tips = sns.load_dataset('tips')
titanic = sns.load_dataset('titanic')

Mentor qeydi: Burada stats subkitabxanası sizin işinizdə yardımçı olacaq

## Hissə 1: Etibarlılıq İntervalları (Confidence Intervals)

### Sual 1: Mean
Nöqtəvi qiymətləndirmə (Point Estimate) tək bir rəqəmdən ibarətdir. `tips` datasetindən istifadə edərək ümumi hesabın (`total_bill`) nümunə ortalamasını tapın. Bu ortalama bütün müştərilərin ödədiyi hesab üçün bir nöqtəvi qiymətləndirmədir.

In [43]:
mean_total_bill = tips["total_bill"].mean()
print("Mean Total Bill:", np.round(mean_total_bill, 4))

Mean Total Bill: 19.7859


### Sual 2: Standard Error of the Mean
`tips` datasetindəki `tip` (bəxşiş) sütunu üçün standart səhvi hesablayın. Nəzərə alın ki, standart səhv, nöqtəvi qiymətləndirmənin standart meylidir.

In [ ]:
# Method 1: Scipy Statistics Modul
std_err_tip = stats.sem(tips["tip"])              # Standard error of the mean
print("Standard Error Tip's Mean:", np.round(std_err_tip, 3))

# Method 2: Custom Method
std_dev_tip = tips["tip"].std()
count_tip = tips["tip"].count()
std_err_tip = std_dev_tip/np.sqrt(count_tip)     # Standard error of the mean
print("Standard Error Tip's Mean:", np.round(std_err_tip, 3))

Standard Error Tip's Mean: 0.089
Standard Error Tip's Mean: 0.089


### Sual 3: Confidence Interval
Populyasiyanın standart meyli naməlum olduqda t-paylanmasından istifadə edilir. T-paylanmasını tətbiq edərək `total_bill` üçün 95%-lik etibarlılıq intervalını hesablayın və nəticəni çap edin.

In [2]:
# Method 1: Custom Method
level_of_significance = 0.05                            # We find alpha in order to use in t-table (with two tail /2 will used)
degree_of_freedom = tips["total_bill"].count() - 1      # We find degree of freedom in order to use in t-table
t_critical = stats.t.ppf(q=1 - level_of_significance/2, df=degree_of_freedom)   # T-Criticals
std_err_tip = stats.sem(tips["tip"])                    # Standard Error of the mean
mean_tip = tips["tip"].mean()                           # Mean

ci_lower_bound = mean_tip - t_critical * std_err_tip
ci_upper_bound = mean_tip + t_critical * std_err_tip
print(f"Confidence Interval: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval: 2.82 - 3.17


In [3]:
# Method 2: With Scipy Statistics
ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence= 1 - level_of_significance, 
    df=degree_of_freedom, 
    loc=mean_tip, 
    scale=std_err_tip
)
print(f"Confidence Interval: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval: 2.82 - 3.17


### Sual 4: Compare ci=99% and ci=95%
Həm `total_bill`, həm də `tip` üçün 99% etibarlılıq intervalını hesablayın və ilkin 95%-lik intervalla müqayisə edin. Etibarlılıq səviyyəsi artdıqca intervalın genişliyi necə dəyişir?

In [5]:
# confidence interval: 95%
ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=0.95, 
    df=degree_of_freedom, 
    loc=mean_tip, 
    scale=std_err_tip
)
print(f"Confidence Interval - 95%: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

# confidence interval: 99%
ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=0.99, 
    df=degree_of_freedom, 
    loc=mean_tip, 
    scale=std_err_tip
)
print(f"Confidence Interval - 99%: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval - 95%: 2.82 - 3.17
Confidence Interval - 99%: 2.77 - 3.23


#### Final Insights:
When we increase the confidence interval from 95% to 99% the result will be wider.

### Sual 5: Degree of Freedom and T-critcs
`titanic` datasetində sərnişinlərin yaşları (`age`) üçün sərbəstlik dərəcəsini (n - 1) tapın və 90% etibarlılıq üçün tələb olunan t-kritik dəyərini hesablayın.

In [6]:
degree_of_freedom = titanic["age"].count() - 1
confidence_interval = 0.90
level_of_significance = 1 - confidence_interval

t_critic = stats.t.ppf(q=1 - level_of_significance/2, df=degree_of_freedom)
print(f"T-critics: -{t_critic:.3f} and +{t_critic:.3f}")

T-critics: -1.647 and +1.647


### Sual 6: Sample Size
`total_bill` ortalamasını ±2 dollar marja xətası daxilində tapmaq üçün tələb olunan minimum seçmə həcmini (n) hesablayın. Populyasiya standart meyli naməlum olduğu üçün, pilot nümunə kimi mövcud datasetin standart meylindən istifadə edin.

In [39]:
error_of_margin = 2
std_dev_total_bill = tips["total_bill"].std()
level_of_significance = 0.05 # For confidence interval 95%
z_critic = stats.norm.ppf(q=1-level_of_significance/2)
sample_size = np.ceil((z_critic**2 * std_dev_total_bill**2) / error_of_margin**2)
print(f"With 95% confindence interval in order to get 2 error of margin we need sample size: {int(sample_size)}")

With 95% confindence interval in order to get 2 error of margin we need sample size: 77


### Sual 7: Managerial Conclusion
Etibarlılıq intervalının qiymətləndirilməsini hesabatda təqdim edərkən seçmə həcmi, etibarlılıq səviyyəsi və nəticənin şərhi mütləq daxil edilməlidir. Yalnız qadın `titanic` sərnişinlərinin yaşları üçün 95% etibarlılıq intervalı quraraq bu qaydaya tam cavab verən kiçik bir hesabat mətni yazın.

In [18]:
mask_woman = titanic["who"] == "woman"
titanic[mask_woman].dropna(subset="age")["age"].count(), titanic[mask_woman].dropna(subset="age")["age"].mean()

(np.int64(218), np.float64(32.0))

In [14]:
mask_woman = titanic["who"] == "woman"    # Filtering only women
titanic_sample = titanic[mask_woman].dropna(subset="age").sample(n=100, random_state=67) # Drop null ages and selecting random sample 100 women age

confidence_interval = 0.95
degree_of_freedom = titanic_sample["age"].count() - 1
mean_age = titanic_sample["age"].mean()
std_err_age = stats.sem(titanic_sample["age"])

ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=confidence_interval, 
    df=degree_of_freedom, 
    loc=mean_age, 
    scale=std_err_age
)

print(f"Confidence Interval: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval: 30.93 - 35.68


#### Final insight:

In Titanic dataset we have 218 women who we now the age. We randomly select sample with size 100 women ages. According to the requirements we conduct Confidence Interval research with 95% confidence rate we get 30.93 and 35.68 interval for women ages within titanic dataset.

### Sual 8: 
Populyasiya normal paylanmadıqda böyük seçmə həcmindən istifadə edilməlidir. `titanic` datasetindəki bilet qiymətləri (`fare`) üçün 95% etibarlılıq intervalı qurun.

In [63]:
titanic_sample = titanic.sample(n=500, random_state=67)
confidence_interval = 0.95
degree_of_freedom = titanic_sample["fare"].count() - 1
mean_fare = titanic_sample["fare"].mean()
std_err_fare = stats.sem(titanic_sample["fare"])

ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=confidence_interval, 
    df=degree_of_freedom, 
    loc=mean_fare, 
    scale=std_err_fare
)

print(f"Confidence Interval: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval: 28.97 - 38.08


### Sual 9: 
Yalnız nahar (Lunch) vaxtı `tips` datasetində ödənilən hesabların ortalaması üçün 95% etibarlılıq intervalı qurun və şərhi qeyd edin.

In [64]:
tips["time"].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [73]:
mask_lunch = tips["time"] == "Lunch"    # Filtering only Lunch
tips_sample = tips[mask_lunch]

confidence_interval = 0.95
degree_of_freedom = tips_sample["total_bill"].count() - 1
mean_bill = tips_sample["total_bill"].mean()
std_err_bill = stats.sem(tips_sample["total_bill"])

ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=confidence_interval, 
    df=degree_of_freedom, 
    loc=mean_bill, 
    scale=std_err_bill
)

print(f"Confidence Interval: {ci_lower_bound:.2f} - {ci_upper_bound:.2f}")

Confidence Interval: 15.30 - 19.04


#### Final Insights:
According to the tips data we can see that in the lunch time we have 68 datapoint which we can get total bill. With 95% confidence interval we calculate mean of the total bill in the lunch meals are within this confidence interval 15.30 - 19.04

## Hissə 2: Hipotezlərin Yoxlanılması (Hypothesis Testing)

### Sual 11: (Step 1: State Null and Alternate Hypothesis)
Hipotez, populyasiya parametri haqqında iddiadır. Bir restoran meneceri `tips` datasetində orta hesabın 20$ olduğunu iddia edir. Bu iddianı yoxlamaq üçün Sıfır (H0) və Alternativ (H1) hipotezlərini Python şərhlərində (comments) necə qurarsınız?

In [ ]:
# Null Hypothesis (H0): population mean = 20
# Alternate Hypothesis (H1): population mean != 20

# This hypothesis is two-tail hypothesis

### Sual 12: (Step 2: Calculate Statistics)
Populyasiya standart meyli naməlum olduqda Z-paylanmasının əvəzinə t-paylanmasından (t-test) istifadə etməlisiniz. 11-ci sualdakı H0 iddiyası üçün t-statistikasını hesablayın.

In [165]:
# Calculate statistics when population standard deviation is unknown:
pop_mean_H0 = 20 # This th manager's hypothesis
sample_mean = tips["total_bill"].sample(n=100, random_state=1234).mean() # We assume that manager just see 100 data as sample and decide 20$ is best match
sample_std_err = stats.sem(tips["total_bill"].sample(n=100, random_state=1234))
degree_of_freedom = tips["total_bill"].sample(n=100, random_state=1234).count() - 1
t_stat = (sample_mean - pop_mean_H0) / sample_std_err
print(f"T-Statistics: {t_stat:.3f}")


T-Statistics: 1.474


### Sual 13: (Step 3a: Consider Decision with P-value) 
P-dəyəri < α olarsa, Sıfır hipotezi (H0) rədd edilir. 12-ci sualdakı nəticəyə əsasən p-dəyərini tapın və α=0.05 səviyyəsində qəti qərarınızı verin.

In [171]:
p_value = stats.t.sf(x=abs(t_stat), df=degree_of_freedom) * 2
level_of_significance = 0.05
print(f"P-Value:                {p_value:.2f}")
print(f"Level of Significance:  {level_of_significance}")
print(f"P-Value({p_value:.2f}) > Level of Significance({level_of_significance})")

P-Value:                0.14
Level of Significance:  0.05
P-Value(0.14) > Level of Significance(0.05)


### Sual 13: (Step 3b: Consider Decision with T-Critics)

In [172]:
t_crit = stats.t.ppf(1 - level_of_significance / 2, df=degree_of_freedom)
print(f"T-Statistics:   {t_stat:.2f}")
print(f"T-Critics:      -{t_crit:.2f} and +{t_crit:.2f}")
print(f"-{t_crit:.2f} < {t_stat:.2f} < +{t_crit:.2f}")

T-Statistics:   1.47
T-Critics:      -1.98 and +1.98
-1.98 < 1.47 < +1.98


In [185]:
tips["total_bill"].mean() # Actual mean of the population

np.float64(19.78594262295082)

#### Managerial Conclusion:
There is enough evidence to conclude that at 5% level of significance (Type I error) manager's option 20$ is not much different actual true mean. As a conclusion we not reject Null Hypothesis.

### Sual 14: (Step 1: State Null and Alternate Hypothesis)
Başqa bir ofisiant iddia edir ki, orta bəxşiş 2.50$-dan çoxdur (H1: μ > 2.50). Bu, bir tərəfli testdir və alternativ hipotez status-kvoya qarşı çıxır. Bu iddia üçün test statistikasını və p-dəyərini hesablayın.

In [161]:
# H0: mean tips <= 2.50$   This is null hypothesis if it is true waiter is false
# H1: mean tips > 2.50$    This is Alternate hypothesis if it is true waiter is true

# This hypothesis test is one-tail test

### Sual 14: (Step 2: Calculate Statistics)

In [179]:
pop_mean_H0 = 2.50   # This is null hypothesis
sample_mean = tips["tip"].sample(n=50, random_state=1234).mean()
sample_std_err = stats.sem(tips["tip"].sample(n=50, random_state=1234))
degree_of_freedom = tips["tip"].sample(n=50, random_state=1234).count() - 1
t_stat = (sample_mean - pop_mean_H0) / sample_std_err
print(f"T-Statistics: {t_stat:.2f}")

T-Statistics: 2.23


### Sual 14: (Step 3a: Consider Decision with P-Value)

In [181]:
p_value = stats.t.sf(x=t_stat, df=degree_of_freedom)
level_of_significance = 0.05
print(f"P-Value:                {p_value:.2f}")
print(f"Level of Significance:  {level_of_significance}")
print(f"P-Value({p_value:.2f}) > Level of Significance({level_of_significance})")

P-Value:                0.02
Level of Significance:  0.05
P-Value(0.02) > Level of Significance(0.05)


### Sual 14: (Step 3b: Consider Decision with T-Critics)

In [182]:
t_crit = stats.t.ppf(1 - level_of_significance, df=degree_of_freedom)
print(f"T-Statistics:   {t_stat:.2f}")
print(f"T-Critics:      -{t_crit:.2f} and +{t_crit:.2f}")
print(f"-{t_crit:.2f} < {t_stat:.2f} < +{t_crit:.2f}")

T-Statistics:   2.23
T-Critics:      -1.68 and +1.68
-1.68 < 2.23 < +1.68


#### Managerial Conclusion:
There is not enough evidence at the 5% level of significance to say that tips is less than 2.50$. As a conclusion we reject Null Hypothesis and Accept Waiter's hypothesis which is the Alternate in this case. 

In [186]:
tips["tip"].mean() # Actual mean of the tips

np.float64(2.99827868852459)

### Sual 15: Error Types
Sıfır hipotezi (H0) doğru olduğu halda onu rədd etmək I növ xəta (Type I Error) adlanır. `titanic` sərnişinlərinin orta yaşının 28 olması iddiasını (H0: μ = 28) yoxlayarkən, I növ və II növ xətaların praktiki olaraq nə demək olduğunu izah edin.

In [ ]:
# Null Hypothesis (H0): mean age = 28
# Alternate Hypothesis (H1): mean age != 28
# 
# Type I error = False Positive (We Incorrectly Reject True Null Hypothesis)
# Type II error = False Negative (We Incorrectly Not Reject False Null Hypothesis)
# 
# Level of Significance = Type I error
# We try to minimize Type II error in order to maximize Power of the Hypothesis Test
# alpha = Type I error, betta = Type II error, 1-betta = Power of the test

In [231]:
sample_ages = titanic["age"].dropna().sample(n=200, random_state=1234)

In [240]:
# Step 1: State NULL and Alternate Hypothesis
# H0: mean age = 28
# H1: mean age != 28
# NOTE: Two-tail test

# Step 2: Calculate Statistics
mean_age_H0 = 28 # Null Hypothesis if True the population mean
sample_mean = sample_ages.mean() # Selected sample mean
sample_std_err = stats.sem(sample_ages) # selected sample standard error
degree_of_freedom = sample_ages.count() - 1 # degree of freedom
t_stat = (sample_mean - mean_age_H0) / sample_std_err
print("H0: avg age = 28\nH1: avg age != 28")
print(f"Sample Mean:    {sample_mean:.2f}")
print(f"T-Statistics:    {t_stat:.3f}")

# Step 3a: Consider Decision Rule with p-value
p_value = stats.t.sf(x=abs(t_stat), df=degree_of_freedom) * 2
level_of_significance = 0.05 # Type I error chance
print(f"Level of Significance:  {level_of_significance:.3f}")
print(f"P-Value:    {p_value:.3f}")

# Step 3b: Consider Decision Rule with t-critics
t_critic = stats.t.ppf(q=1 - level_of_significance/2, df=degree_of_freedom)
print(f"T-Critics:  -{t_crit:.3f} and +{t_crit:.3f}")

# Step 4: Menegerial Conclusion
print(f"P-Value({p_value:.3f}) > Level of Significance({level_of_significance:.3f}):    {p_value > level_of_significance}")
print(f"Left Critical Value (-{t_crit:.2f}) < T-Sttistics ({t_stat:.2f}) < Right Critical Valu (+{t_crit:.2f}): {(t_stat > t_crit*-1) and t_stat < t_crit}")

if p_value > level_of_significance:
    print(f""" 
        Managerial Conclusion of Hypthesis Test:
        We get sample mean {sample_mean:.2f} age also we have NOT enough evidence to REJECT NULL Hypothesis 
        at 5% level of significance level. Average age for the Titanic Data is NOT different from 28
        """)
else:
    print(f""" 
        Managerial Conclusion of Hypthesis Test:
        We get sample mean {sample_mean:.2f} age as well as we have enough evidence to REJECT NULL Hypothesis 
        at 5% level of significance level. Average age for the Titanic Data is different from 28
        """)


H0: avg age = 28
H1: avg age != 28
Sample Mean:    30.01
T-Statistics:    1.993
Level of Significance:  0.050
P-Value:    0.048
T-Critics:  -1.677 and +1.677
P-Value(0.048) > Level of Significance(0.050):    False
Left Critical Value (-1.68) < T-Sttistics (1.99) < Right Critical Valu (+1.68): False
 
        Managerial Conclusion of Hypthesis Test:
        We get sample mean 30.01 age as well as we have enough evidence to REJECT NULL Hypothesis 
        at 5% level of significance level. Average age for the Titanic Data is different from 28
        


Type I error = False Positive = We incorrectly reject TRUE null hypothesis
Type II error = False Negative = We incorrectly NOT reject FALSE null hypothesis

The Level of Significance = Type I error probablity
If we get p-value < the level of significance it means our decision can be False Positive which means we incorrectly reject TRUE null hypothesis

### Sual 16: T-Critics
Kritik dəyər test statistikasının qərar qəbuletmə sərhədini müəyyən edir. `tips` datasetində kişi müştərilərin verdiyi hesabın orta dəyərinin 20$ olub-olmadığını α=0.05 səviyyəsi üçün kritik dəyər yanaşması (Critical Value Approach) ilə yoxlayın.

In [258]:
# Sample Data
sample_data = tips[tips["sex"] == "Male"]["total_bill"].sample(n=20, random_state=1234)

# Hypthesis Testing
# Step 1: State Null and Alternate Hypothesis
# NULL Hypothesis H0: mean bill = 20$
# Alternate Hypothesis H1: mean bill != 20$
# NOTE: This test will be two-tail test

# Step 2: Calculate Statistics
mean_bill_H0 = 20
level_of_significance = 0.05
sample_mean = sample_data.mean()
sample_std_err = stats.sem(sample_data)
degree_of_freedom = sample_data.count() - 1

t_stat = (sample_mean - mean_bill_H0) / sample_std_err
print(f"T-Statistics:   {t_stat:.3f}")

# Step 3: Conclude Decision with T-Critics
t_crit = stats.t.ppf(q=1 - level_of_significance/2, df=degree_of_freedom)
print(f"T-Critics:  -{t_crit:.3f} and +{t_crit:.3f}")

# Step 4: Conclusion
print(f"Left T-Critic (-{t_crit:.3f}) < T-Statistics ({t_stat:.3f}) < Right T-Critic (+{t_crit:.3f})")
print(f"""
      Conclusion:
      We have NOT enough evidance at 5% level of significance to REJECT NULL Hypothesis.
      As a result of Hypothesis Test Average Bill of Male Customers is NOT differ from 20$ 
""")

T-Statistics:   0.946
T-Critics:  -2.093 and +2.093
Left T-Critic (-2.093) < T-Statistics (0.946) < Right T-Critic (+2.093)

      Conclusion:
      We have NOT enough evidance at 5% level of significance to REJECT NULL Hypothesis.
      As a result of Hypothesis Test Average Bill of Male Customers is NOT differ from 20$ 



In [259]:
sample_mean # Our selected sample's mean

np.float64(22.110500000000002)

In [260]:
tips[tips["sex"] == "Male"]["total_bill"].mean() # Actual population data mean

np.float64(20.744076433121016)

### Sual 17: Reject or NOT Reject H0
H0-ı rədd etmək, məlumatların α səviyyəsində H1-in xeyrinə əhəmiyyətli sübutlar təqdim etdiyini göstərir. `titanic` sərnişinlərinin bilet qiymətlərinin (`fare`) 35$-dan əhəmiyyətli dərəcədə fərqli olub-olmadığını yoxlayın və idarəetmə baxımından nəticəni şərh edin.

In [263]:
# Data
sample_data = titanic["fare"].sample(n=20, random_state=1234)

# Hypthesis Testing
# Step 1: State Null and Alternate Hypothesis
# NULL Hypothesis H0: mean bill = 20$
# Alternate Hypothesis H1: mean bill != 20$
# NOTE: This test will be two-tail test

# Step 2: Calculate Statistics
mean_fate_H0 = 20
level_of_significance = 0.05
sample_mean = sample_data.mean()
sample_std_err = stats.sem(sample_data)
degree_of_freedom = sample_data.count() - 1

t_stat = (sample_mean - mean_bill_H0) / sample_std_err
print(f"T-Statistics:   {t_stat:.3f}")

# Step 3: Conclude Decision with T-Critics
t_crit = stats.t.ppf(q=1 - level_of_significance/2, df=degree_of_freedom)
print(f"T-Critics:  -{t_crit:.3f} and +{t_crit:.3f}")

# Step 4: Conclusion
print(f"Left T-Critic (-{t_crit:.3f}) < T-Statistics ({t_stat:.3f}) < Right T-Critic (+{t_crit:.3f})")
print(f"""
      Conclusion:
      We have NOT enough evidance at 5% level of significance to REJECT NULL Hypothesis.
      As a result of Hypothesis Test Average Ticket Price is NOT differ from 35$ 
""")

T-Statistics:   -0.170
T-Critics:  -2.093 and +2.093
Left T-Critic (-2.093) < T-Statistics (-0.170) < Right T-Critic (+2.093)

      Conclusion:
      We have NOT enough evidance at 5% level of significance to REJECT NULL Hypothesis.
      As a result of Hypothesis Test Average Ticket Price is NOT differ from 35$ 



In [265]:
sample_mean

np.float64(19.241049999999998)

In [266]:
titanic["fare"].mean()

np.float64(32.204207968574636)

### Sual 18: Reject or Not Reject H0
Həftəsonu verilən bəxşişlərin (tip) ortalamasının 3 dollara bərabər olması iddiasını yoxlayın. Əgər test statistikası qəbul bölgəsinə (non-rejection region) düşərsə, H0-ı rədd etməyin. Qərarınızı qeyd edin.

In [40]:
# Data
#-----------------------
sample_data = tips[tips["day"].isin(values=["Sat", "Sun"])]["tip"].sample(n=25, random_state=1234)

# Hypothesis Testing
#------------------------
# Step 1: State NULL and Alternate Hypothesis
# NULL Hypothesis (H0): mean tip = 3$
# Alternate Hypothesis (H1): mean tip != 3$
# NOTE: Two-tailed Test

# Step 2: Select Level of Significance
level_of_significance = 0.05

# Step 3: Calculate Test Statistics
null_hypothesis = 3
sample_mean = sample_data.mean()
sample_std_err = stats.sem(sample_data)
degree_of_freedom = sample_data.count() - 1

t_stat = (sample_mean - null_hypothesis) / sample_std_err
print(f"T-Statistics:   {t_stat:.3f}")

# Step 4a: Consider Decision with T-Critics
t_crit = stats.t.ppf(q=1-level_of_significance/2, df=degree_of_freedom)
print(f"T-Critics:  -{t_crit:.3f} and +{t_crit:.3f}")

flag_critics = -1*t_crit < t_stat < t_crit

# Step 4b: Consider Decision ith P-Value
p_value = stats.t.sf(x=abs(t_stat), df=degree_of_freedom) * 2

flag_p = p_value > level_of_significance

# Step 5: Make decision
print(f"P-Value({p_value:.3f}) > Level of Significance({level_of_significance:.3f}):    {flag_p}")
print(f"Left Critical Value (-{t_crit:.3f}) < T-Sttistics ({t_stat:.3f}) < Right Critical Value (+{t_crit:.3f}): {flag_critics}")

T-Statistics:   0.231
T-Critics:  -2.064 and +2.064
P-Value(0.820) > Level of Significance(0.050):    True
Left Critical Value (-2.064) < T-Sttistics (0.231) < Right Critical Value (+2.064): True


In [10]:
sample_mean

np.float64(3.0652000000000004)

In [11]:
tips[tips["day"].isin(values=["Sat", "Sun"])]["tip"].mean()

np.float64(3.115276073619632)

### Sual 19: Confidence Interval vs Level of Significance
İki tərəfli testlərin nəticələri etibarlılıq intervalları ilə əlaqəlidir. `tips` datasetində ümumi hesablar üçün 95% etibarlılıq intervalı qurun. Əgər yoxlanılan iddia dəyəri bu intervalın daxilində deyilsə, H0 hipotezinin α=0.05 səviyyəsində rədd edildiyini praktik olaraq sübut edin.

In [47]:
# Confidence Interval According to the Z-test
ci_lower_bound, ci_upper_bound = stats.norm.interval(
    confidence=0.95,
    loc=tips["total_bill"].mean(),
    scale=stats.sem(tips["total_bill"])
)
print(f"95% Confidence Interval (Z-test): ({ci_lower_bound:.2f}, {ci_upper_bound:.2f})")

95% Confidence Interval (Z-test): (18.67, 20.90)


In [54]:
# Confidence Interval According to the t-test
ci_lower_bound, ci_upper_bound = stats.t.interval(
    confidence=0.95,
    df=19,
    loc=tips["total_bill"].sample(n=20, random_state=1234).mean(),
    scale=stats.sem(tips["total_bill"].sample(n=20, random_state=1234))
)
print(f"95% Confidence Interval (T-test): ({ci_lower_bound:.2f}, {ci_upper_bound:.2f})")

95% Confidence Interval (T-test): (16.73, 25.35)


In [ ]:
# Data
sample_data = tips["total_bill"].sample(n=20, random_state=1234)

# Hypothesis Testing
#------------------------
# Step 1: State NULL and Alternate Hypothesis
# NULL Hypothesis (H0): mean bill = 23$
# Alternate Hypothesis (H1): mean tip != 23$
# NOTE: Two-tailed Test

# Step 2: Select Level of Significance
level_of_significance = 0.05

# Step 3: Calculate Test Statistics
null_hypothesis = 23
sample_mean = sample_data.mean()
sample_std_err = stats.sem(sample_data)
degree_of_freedom = sample_data.count() - 1

t_stat = (sample_mean - null_hypothesis) / sample_std_err
print(f"T-Statistics:   {t_stat:.3f}")

# Step 4a: Consider Decision with T-Critics
t_crit = stats.t.ppf(q=1-level_of_significance/2, df=degree_of_freedom)
print(f"T-Critics:  -{t_crit} and +{t_crit}")

flag_critics = -1*t_crit < t_stat < t_crit

# Step 4b: Consider Decision ith P-Value
p_value = stats.t.sf(x=abs(t_stat), df=degree_of_freedom) * 2
print(f"P-Value:    {p_value:.3f}")

flag_p = (p_value > level_of_significance) and (p_value <= 1)

# Step 5: Make decision
print(f"P-Value({p_value:.3f}) > Level of Significance({level_of_significance:.3f}):    {flag_p}")
print(f"Left Critical Value (-{t_crit:.3f}) < T-Sttistics ({t_stat:.3f}) < Right Critical Valu (+{t_crit:.3f}): {flag_critics}")

T-Statistics:   -0.952
T-Critics:  -2.0930240544083087 and +2.0930240544083087
P-Value(0.353) > Level of Significance(0.050):    True
Left Critical Value (-2.093) < T-Sttistics (-0.952) < Right Critical Valu (+2.093): True


### Sual 20: Hypothesis Test Full Workflow
Hipotezin yoxlanılması prosesinin ardıcıl 6 addımını `titanic` gəmisində 3-cü sinif bilet qiymətlərinin ortalamasının 15$-a bərabər olması iddiası üzərində tətbiq edərək tam bir nümunə həlli təqdim edin.

In [69]:
# Data
sample_data = titanic[titanic["class"] == "Third"]["fare"].sample(n=25, random_state=1234)
#------------------

# Hypothesis Testing Workflow
#-------------------------------
# Step 1: State NULL and Alternate Hypothesis
# NULL Hypothesis (H0): mean price = 15$
# Alternate Hypothesis (H1): mean price != 15$
# NOTE: Two-tailed Test 
#--------------------------------------------

# Step 2: Select Level of Significance
level_of_significance = 0.05
#-------------------------------------

# Step 3: Calculate Test Statistics (T-test or Z-test)
null_hypothesis = 15
sample_mean = sample_data.mean()
sample_std_err = stats.sem(sample_data)
degree_of_freedom = sample_data.count() - 1

t_stat = (sample_mean - null_hypothesis) / sample_std_err
print(f"T-statistics:   {t_stat:.3f}")
#----------------------------------

# Step 4a: Consider Decision with P-value
p_value = stats.t.sf(x=abs(t_stat), df=degree_of_freedom) * 2
print(f"P-Value:    {p_value:.3f}")

flag_p = (p_value > level_of_significance) and (p_value <= 1)
#---------------------------------------- 

# Step 4b: Consider Decision with T-critics
t_crit = stats.t.ppf(q=1-level_of_significance/2, df=degree_of_freedom)
print(f"T-critics:  -{t_crit:.3f} and +{t_crit:.3f}")

flag_critics = -1*t_crit < t_stat < t_crit
#-------------------------------------------

# Step 5: Managerial Conclusion:
print(f"P-Value({p_value:.3f}) > Level of Significance({level_of_significance:.3f}):    {flag_p}")
print(f"Left Critical Value (-{t_crit:.3f}) < T-Sttistics ({t_stat:.3f}) < Right Critical Valu (+{t_crit:.3f}): {flag_critics}")

T-statistics:   -0.363
P-Value:    0.720
T-critics:  -2.064 and +2.064
P-Value(0.720) > Level of Significance(0.050):    True
Left Critical Value (-2.064) < T-Sttistics (-0.363) < Right Critical Valu (+2.064): True


#### Managerial Conclision
We get approximately 14 average ticket price as well as we have NOT enough evidance to REJECT NULL hypothesis. As a result average ticket price within third class is not different 15$.

In [66]:
titanic[titanic["class"] == "Third"]["fare"].mean()

np.float64(13.675550101832993)

In [67]:
titanic[titanic["class"] == "Third"]["fare"].sample(n=25, random_state=1234).mean()

np.float64(13.918168000000001)